In [47]:
import numpy as np

In [48]:
import pandas as pd

In [49]:
dataset = pd.read_csv("Social_Network_Ads.csv")

In [50]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [51]:
dataset = pd.get_dummies(dataset, dtype=int, drop_first = True)

In [52]:
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [53]:
dataset = dataset.drop("User ID", axis = 1)

In [54]:
dataset

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [55]:
dataset.columns

Index(['Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [56]:
independent = dataset[["Age", "EstimatedSalary", "Gender_Male"]]

In [57]:
independent

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1
...,...,...,...
395,46,41000,0
396,51,23000,1
397,50,20000,0
398,36,33000,1


In [58]:
dependent = dataset[["Purchased"]]

In [59]:
dependent

,Purchased
0,0
1,0
2,0
3,0
4,0
...,...
395,1
396,1
397,1
398,0


In [60]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train, y_test = train_test_split(independent, dependent, test_size =0.30, random_state = 0)

In [61]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [62]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
param_grid = {'kernel':['linear','rbf','poly','sigmoid'],'gamma':['auto','scale'],'C':[10,100,1000,2000,3000]}
grid = GridSearchCV(SVC(probability = True), param_grid, refit = True, verbose = 3, cv =5, n_jobs=-1, scoring='f1_weighted')
grid.fit(x_train,np.ravel(y_train))
re=grid.cv_results_
grid_predictions = grid.predict(x_test)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


In [63]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,grid_predictions)

In [64]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [65]:
#from sklearn.metrics import f1_score
#f1_macro=f1_score(y_test, grid_predictions, average = 'weighted')
#print("The f1_macro value for best parameter{}:".format(grid.best_params_),f1_macro)
print("The best parameter set for this model:\n", format(grid.best_params_))
print("The Confusion Matrix:\n",cm)
print("The report:\n", clf_report)
from sklearn.metrics import roc_auc_score
roc_score = roc_auc_score(y_test, grid.predict_proba(x_test)[:,1])
print("The ROC_Score for this model:\n", roc_score)

The best parameter set for this model:
 {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
The Confusion Matrix:
 [[73  6]
 [ 4 37]]
The report:
               precision    recall  f1-score   support

           0       0.95      0.92      0.94        79
           1       0.86      0.90      0.88        41

    accuracy                           0.92       120
   macro avg       0.90      0.91      0.91       120
weighted avg       0.92      0.92      0.92       120

The ROC_Score for this model:
 0.9654214263661625


In [66]:
table = pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.022213,0.002888,0.005902,0.000782,10,auto,linear,"{'C': 10, 'gamma': 'auto', 'kernel': 'linear'}",0.835985,0.782971,0.644599,0.927778,0.890114,0.816289,0.098849,21
1,0.013030,0.002199,0.005899,0.000713,10,auto,rbf,"{'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}",0.892857,0.875644,0.841398,0.947015,0.946153,0.900613,0.041029,1
2,0.019015,0.002927,0.005569,0.000632,10,auto,poly,"{'C': 10, 'gamma': 'auto', 'kernel': 'poly'}",0.833024,0.799537,0.737557,0.928571,0.888158,0.837369,0.066798,12
3,0.010193,0.001028,0.004935,0.000869,10,auto,sigmoid,"{'C': 10, 'gamma': 'auto', 'kernel': 'sigmoid'}",0.769053,0.733523,0.730543,0.753871,0.759910,0.749380,0.014996,32
4,0.017665,0.001425,0.004357,0.000251,10,scale,linear,"{'C': 10, 'gamma': 'scale', 'kernel': 'linear'}",0.835985,0.782971,0.644599,0.927778,0.890114,0.816289,0.098849,21
5,0.012425,0.000851,0.005509,0.000982,10,scale,rbf,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.892857,0.875644,0.841398,0.947015,0.946153,0.900613,0.041029,1
6,0.016441,0.001724,0.004469,0.000268,10,scale,poly,"{'C': 10, 'gamma': 'scale', 'kernel': 'poly'}",0.833024,0.799537,0.737557,0.928571,0.888158,0.837369,0.066798,12
7,0.011411,0.000427,0.004815,0.000185,10,scale,sigmoid,"{'C': 10, 'gamma': 'scale', 'kernel': 'sigmoid'}",0.769053,0.733523,0.610390,0.753871,0.759910,0.725349,0.058652,40
8,0.056431,0.008099,0.005004,0.001722,100,auto,linear,"{'C': 100, 'gamma': 'auto', 'kernel': 'linear'}",0.835985,0.782971,0.644599,0.927778,0.890114,0.816289,0.098849,21
9,0.020312,0.002206,0.005443,0.001068,100,auto,rbf,"{'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}",0.855314,0.892857,0.859435,0.929144,0.928571,0.893064,0.031996,3


In [67]:
Age_input = float(input("Age="))
Est_salary_input = float(input("Estimated Salary="))
Gender_male_input = int(input("Gender Male="))

Age= 70
Estimated Salary= 3000
Gender Male= 1


In [69]:
Future_predictions = grid.predict([[Age_input, Est_salary_input, Gender_male_input]])
print("Future Predictions:\n", Future_predictions)

Future Predictions:
 [1]
